# Encoder + Encoder-Decoder runs — BlackboxNLP

These are the **last open scientific question**: does the U-shaped, residual-controlled
linearity profile replicate outside decoder-only models?

**Run `fit` and `randinit` for all four models FIRST.** Those two stages alone answer the
question. `ppl`/`multi`/`mlp` for BERT/RoBERTa are nice-to-have; run them only if time allows.

Notes:
- These models are small (110M–220M), so each `fit` is ~10–25 min on a T4 — much faster than Qwen/LLaMA.
- **BART and T5 have no perplexity/patching stage by design** (T5 threads position_bias between
  blocks, so a drop-in patch breaks the chain). They contribute R² curves for BOTH stacks —
  including the decoder stacks the SRW paper never analysed.
- BERT/RoBERTa functional metric is **masked-token accuracy** (higher = better), NOT perplexity.
- **Check the GPU is on before starting** (Runtime → Change runtime type → T4).

In [ ]:
!nvidia-smi   # MUST show a GPU. If not: Runtime -> Change runtime type -> T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/blackboxnlp/results"
!mkdir -p $OUT

In [ ]:
!pip -q install transformers==4.46.3 datasets==3.1.0 accelerate pandas matplotlib
!unzip -oq pipeline_encoders.zip -d /content/
%cd /content/pipeline

## Step 1 — the claim-critical runs (do these first, all four)

In [ ]:
!python run.py --model bert-base    --stage fit randinit --out $OUT

In [ ]:
!python run.py --model roberta-base --stage fit randinit --out $OUT

In [ ]:
!python run.py --model bart-base    --stage fit randinit --out $OUT

In [ ]:
!python run.py --model t5-base      --stage fit randinit --out $OUT

## Step 2 — read the answer: does the U-shape replicate?

In [ ]:
import pandas as pd
for m in ["bert-base","roberta-base","bart-base","t5-base"]:
    try:
        df = pd.read_csv(f"{OUT}/{m}/r2.csv")
        print(f"\n=== {m} ===")
        for stack in df["stack"].unique():
            s = df[df["stack"]==stack]
            print(f"  [{stack}] delta R2: first={s.r2_delta.iloc[0]:.3f} "
                  f"mid={s.r2_delta[2:-2].mean():.3f} last={s.r2_delta.iloc[-1]:.3f}"
                  f"  | identity mid={s.r2_identity[2:-2].mean():.3f}"
                  f"  | full mid={s.r2_full[2:-2].mean():.3f}")
        rnd = pd.read_csv(f"{OUT}/{m}/r2_randinit.csv")
        for stack in rnd["stack"].unique():
            s = rnd[(rnd["stack"]==stack) & (rnd.layer < rnd.layer.max())]
            print(f"  [{stack}] RANDOM-INIT delta R2 mean={s.r2_delta.mean():.3f} (flat ~0.18 expected)")
    except FileNotFoundError:
        print(f"{m}: not run yet")

**How to read it:** if `first`/`last` delta-R² are clearly higher than `mid` (as in Qwen/LLaMA),
the U-shape replicates → the profile is a property of trained transformers generally.
If encoders are flat or inverted → the U is decoder-specific, which is a *sharper* claim.
Either way the random-init rows should be flat near 0.18. **No bad outcome — write it up either way.**

## Step 3 — optional (only if time): encoder functional results

In [ ]:
!python run.py --model bert-base    --stage ppl multi mlp --out $OUT

In [ ]:
!python run.py --model roberta-base --stage ppl multi mlp --out $OUT